# FastAPI

source:
- [FastAPI](https://fastapi.tiangolo.com/)
- [Starlette](https://starlette.dev/)
- [ASGI](https://asgi.readthedocs.io/en/latest/)

## Introduction

### Fast API

FastAPI is a framework by tiangolo to build async-first webapps, it's built on top of Starlette and adds:
- Pydantic request/response validation
- Dependency Injection
- Swagger
- Typed parameter parsing

### Starlette

Starlette is an ASGI unopinionated toolkit meant to build async web services. It introduces:
- Routing
- Middleware (Including Exception Handling, Custom handlers, Error middleware)
- Request, response, streaming
- Websockets, background tasks, ASGI's lifespan (app side)
- HTTPException
- TestClient (built on httpx), sessions, cookies, CORS
- NO validation, NO DI, NO Swagger, NO Typed Parameter parsing

### ASGI

ASGI is the standard for async Python web apps and the successor to WSGI (2003), which remains the standard for sync ones.
It ensures that async webservers, middlewares, webapps frameworks written in python are compatible with each other. 
WSGI's model is "take a request, return a response", so it structurally cannot do WebSockets, Server-Sent-Events or streaming: there's no way to send more after returning. ASGI replaces it with "send messages until done".

- ASGI is a standard (spec) that defines a contract between
    - Webapps / Middlewares: Python apps aspiring to be webapps or middlewares (like FastAPI, Starlette, CORSMiddleware)
    - Webservers: Python apps meant to serve webapps (like uvicorn)
- ASGI states that 
    - Webapps / Middleware should be an async callable with three parameters: scope, receive, send
    - Webservers should:
        - Provide a valid scope per connection (path, query_string, raw_path, headers, ...)
        - Call the app exactly once per connection
        - Provide a valid receive, honoring the message ordering 
            - receive is an awaitable callable (backed by a queue) returning at each call a single http.request or http.disconnect 
            - the queue contains one or more http.request (all but the last have more_body=True) 
            - the queue may contain one http.disconnect if the user dropped before the server finished sending its response. awaiting the receive also returns http.disconnect after the response is complete
        - Provide a valid send 
            - accept the messages from the webapp 
            - verify the responses are legal and in the correct order (just one http.response.start, no more responses after more_body=False)
        - Apply backpressure: await send() returns only once data hit the transport (otherwise a slow client on a token stream blows up memory) 
            - Webapp calls await send(chunk) to send part of the response
            - Uvicorn writes to the socket and hits a full buffer (TCP protocol knows how to say "stop, i'm full")
            - Uvicorn await drain()
            - When the socket can accept more, drain() returns and then send(chunk) returns
            - Webapp can call await send(chunk) again to send more of the response  
        - Signal user disconnections to the app (useful to stop paying LLMs token if user is gone)
        - Run the lifespan protocol
            - create a lifespan scope
            - at boot
                - send lifespan.startup
                - wait for lifespan.startup.complete or lifespan.startup.failed
            - the app lives...
            - at shut down
                - send lifespan.shutdown
                - wait for lifespan.shutdown.complete or lifespan.shutdown.failed
            - Note: if the app doesn't support lifespan and raises, catch it, disable lifespan, serve HTTP normally
            - Note: Here you will setup some resources that must be alive for the whole process and can be accessed from every request*
        - Handle app exceptions without crashing the server
        - Serve connections concurrently (many app invocations on one event loop)
- A Callable is everything that can be called (both a function or a class implementing \_\_call\_\_)
- \* Uvicorn shallow-copies the lifespan scope["state"] inside the per-connection scope["state"], but it's an optional part of the ASGI specs. To avoid this optional spec, Starlette provides its own request.app.state.XXX and FastAPI provides a convenient way to wrap this call with DI (in order to mock it or to easily replace the dependency)
```python
    @asynccontextmanager
    async def lifespan(app):
        app.state.llm_model = load_model()
        yield

    def get_llm_model(request: Request):
        return request.app.state.llm_model

    @app.post("/predict")
    async def predict(llm_model = Depends(get_llm_model)):
        return llm_model.encode(...)  
```

Note: 
- ASGI covers only the handoff among server <-> middlewares <-> app. Every additional logic like CORS, Rate-Limit, Auth, Caching, Routing, Path Params, Validation, Session, Background Tasks, Open API docs, are not concerned in the ASGI specs and are features of FastAPI / Starlette / Middlewares. 

Note2:
- Flask and Django's classic path are still WSGI. 
- The payoff is concurrency on I/O-bound work (waiting on LLMs, DBs, sockets) but not on CPU-bound work. A blocking call inside async def stalls every other request.

## FastAPI BIG DONTS

There are two things worth knowing that should be never done in FastAPI:
1. THE EVENT-LOOP TRAP:
    - Waiting on blocking resources or doing intensive CPU work inside an async def endpoint is __A WORST PRACTICE__ 
    - async def endpoints run inside a main loop so if you stall an async def endpoint you'll stall the main loop and so every other requests
    - You have 4 solutions:
        - Use async def endpoints when you have no intensive CPU work and no I/O blocking (async I/O is OK) 
        - Use async def endpoints but run I/O blocking part in await asyncio.to_thread(fn, args) when only a part of the process is I/O blocking
        - Use def endpoints when the endpoint is mostly I/O blocking
            - def endpoints run by default in a Threadpool and so they don't freeze the main loop on I/O blocking
            - but they cannot parallelize CPU work because they run under same process and the same GIL (CPU Work will share the time of the same core of other requests) 
        - Use workers or task queue to parallelize work when CPU intensive work is needed
            - Workers run in uvicorn but on their own separate processes (with their own interpreter and GIL) and so they can run in parallel (use another core)
                - You USE this for fast jobs (less than 30 seconds)
                - They run on a different process but they still are linked to the httprequest life cycle so they become unreliable when response isn't received in 30 seconds (Due to browsers/proxies/loadbalancers timeouts)
                - PAY attention because every worker runs on a different process and has a copy of everything loaded at lifespan so RAM is the limit 
            - Task queues run externally and so they offload the work to other processes, can run in parallel and have additional pros
                - You USE this for long jobs (30+ seconds)
                - Client receives a 202 as soon as the task has been pushed to the queue so queues are not linked to the httprequest life cycle 
                - Queues can manage spikes by keeping the tasks in the queue
                - Queues can retry the work if it fails
                - Queues can be handled by workers on separate machines (horizontal scaling)
        - Note: when you delegate CPU work to c extensions (Numpy, Pytorch, ...), the c code releases the GIL and can work on different cpu cores in parallel with other requests
    - If you use async httpx / async sdks you won't have this problem since the CPU work is done outside by the LLM and you're using a non-blocking I/O. 
    - When using LLMs you use streaming on async def endpoint to give partial results to the client and keep the http request alive 
2. LIFESPAN SCOPED OBJECTS POISONING:
    - Lifespan scoped objects are shared BY REFERENCE across every request
    - Treat lifespan scoped objects as read-only because if you mutate their state you will mutate that state for every request
    - Instead of mutating the state you must pass per-request configs as methods' arguments